# 03 PyTorch Training v1

Persistent PyTorch warm-up + fine-tune test run. This notebook uses the existing NB02 split manifests and `src_torch` helpers; TensorFlow notebooks are not modified.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('MPLCONFIGDIR', '/private/tmp')

print('PROJECT_ROOT:', PROJECT_ROOT)

PROJECT_ROOT: /Users/starsrain/2025_codeProject/GreenSpace_CNN


In [2]:
import torch
import torchvision
import torchgeo

from src_torch.config import TORCH_DATA_CONFIG, TORCH_MODEL_CONFIG, TORCH_TRAINING_CONFIG
from src_torch.training import run_persistent_warmup_finetune

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('torchgeo:', torchgeo.__version__)
print('TORCH_DATA_CONFIG:', TORCH_DATA_CONFIG)
print('TORCH_MODEL_CONFIG:', TORCH_MODEL_CONFIG)
print('TORCH_TRAINING_CONFIG:', TORCH_TRAINING_CONFIG)

torch: 2.10.0
torchvision: 0.25.0
torchgeo: 0.8.1
TORCH_DATA_CONFIG: {'img_size': (512, 512), 'batch_size': 4, 'num_workers': 0, 'pin_memory': False, 'image_transform': 'tf_parity', 'backbone_preprocess': None, 'use_oversampling': True, 'use_augmentation': True, 'oversampling_sanity_batches': 80}
TORCH_MODEL_CONFIG: {'backbone_priority': 'torchgeo', 'torchgeo_model_name': 'resnet50', 'torchgeo_weight': 'ResNet50_Weights.FMOW_RGB_GASSL', 'load_pretrained_weights': True, 'fallback_backbone': 'torchvision', 'num_binary': 7, 'num_shade': 2, 'score_output_range': (1.0, 5.0), 'veg_output_range': (1.0, 5.0)}
TORCH_TRAINING_CONFIG: {'seed': 37, 'test_run_mode': True, 'test_warmup_epochs': 1, 'test_finetune_epochs': 1, 'warmup_epochs': 5, 'finetune_epochs': 100, 'warmup_learning_rate': 0.001, 'finetune_learning_rate': 0.0001, 'fine_tune_backbone': True, 'use_combo_training_control': True, 'combo_w_bin': 0.5, 'combo_w_ord': 0.5, 'combo_mcmae_scale': 2.0, 'early_stopping_patience': 10, 'restore_b

## Run 1+1 Training Test

Expected behavior: one warm-up epoch with the TorchGeo backbone frozen, then one fine-tune epoch with the backbone trainable when `fine_tune_backbone=True`. Artifacts are saved under `models/runs/PyTorch_<timestamp>/`.

In [3]:
# Full PyTorch training run on Apple Silicon MPS.
training_override = {
    'test_run_mode': False,
    'device': 'mps',
    'max_train_batches': None,
    'max_val_batches': None,
}

result = run_persistent_warmup_finetune(training_config=training_override)
result

RUN_TAG: PyTorch_20260531_211840
RUN_DIR: /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_211840
device: mps
test_run_mode=False -> warmup=5, finetune=100
batch caps: train=None, val=None
oversampling plan: {'active': True, 'streams': [{'label': 'children_s_playground_p', 'target_rate': 0.2, 'pos_threshold': 0.5, 'row_count': 381, 'base_soft_mean': 0.1127757852407347, 'threshold_rate': 0.12422562764916857}, {'label': 'water_feature_p', 'target_rate': 0.25, 'pos_threshold': 0.5, 'row_count': 557, 'base_soft_mean': 0.18941419410933596, 'threshold_rate': 0.20052168242582327}], 'remainder_count': 2129, 'remainder_weight': 0.55, 'weights': [0.2, 0.25, 0.55]}
training control: {'monitor': 'val_training_combo', 'mode': 'max', 'early_stopping_patience': 10, 'restore_best_weights': True, 'reduce_lr_factor': 0.5, 'reduce_lr_patience': 2, 'reduce_lr_min_delta': 0.0001, 'fine_tune_mae_guardrail': {'delta': 0.05, 'patience': 10}}
Warm-up trainable summary: {'total_para

{'run_tag': 'PyTorch_20260531_211840',
 'run_dir': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_211840',
 'final_model_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_211840/final_PyTorch_20260531_211840.pt',
 'best_mc_mae_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_211840/best_mcmae_PyTorch_20260531_211840.pt',
 'best_prauc_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_211840/best_prauc_PyTorch_20260531_211840.pt',
 'model_config_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_211840/model_config_PyTorch_20260531_211840.json',
 'training_history_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_211840/training_history_PyTorch_20260531_211840.json',
 'training_curves_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_211840/tra

In [4]:
run_dir = Path(result['run_dir'])
print('RUN_DIR:', run_dir)
for path in sorted(run_dir.iterdir()):
    print(path.name)

RUN_DIR: /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_211840
best_mcmae_PyTorch_20260531_211840.pt
best_prauc_PyTorch_20260531_211840.pt
final_PyTorch_20260531_211840.pt
model_config_PyTorch_20260531_211840.json
training_curves.png
training_history_PyTorch_20260531_211840.json
